# Semana 5: KNN, Naive Bayes y SVM
## Notebook de Ejercicios (NB2) – Aplicación a Datos Reales

**Propósito:** Aplicar los algoritmos de clasificación clásicos (KNN, Naive Bayes, SVM) a datasets reales, optimizar sus hiperparámetros y comparar su rendimiento.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Clasificar dígitos escritos a mano con KNN y SVM.
- Clasificar textos con Multinomial Naive Bayes.
- Optimizar hiperparámetros mediante validación cruzada (GridSearchCV).
- Comparar tiempos de inferencia entre KNN y SVM.
- Evaluar y comparar el rendimiento de los diferentes algoritmos.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y fijamos la semilla para reproducibilidad.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

# Scikit-learn - Datasets
from sklearn.datasets import load_digits, fetch_20newsgroups

# Preprocesamiento
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

# Modelos
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.svm import SVC

# Métricas
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Semilla
np.random.seed(42)

print("Librerías importadas correctamente.")

---
# PARTE 1: CLASIFICACIÓN DE DÍGITOS CON KNN Y SVM

El dataset **Digits** contiene imágenes de 8x8 píxeles de dígitos escritos a mano (0-9). Es un problema de clasificación multiclase con 10 clases.

## 1. Carga y Exploración del Dataset Digits

In [ ]:
# Cargamos el dataset
digits = load_digits()
X_digits, y_digits = digits.data, digits.target

print(f"Dimensiones de X_digits: {X_digits.shape}")
print(f"Dimensiones de y_digits: {y_digits.shape}")
print(f"Número de clases: {len(np.unique(y_digits))}")
print(f"Clases: {np.unique(y_digits)}")

# Mostramos algunas imágenes de ejemplo
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(digits.images[i], cmap='gray')
    ax.set_title(f'Dígito: {digits.target[i]}')
    ax.axis('off')
plt.suptitle('Ejemplos de dígitos del dataset')
plt.tight_layout()
plt.show()

## 2. Preprocesamiento y División Train/Test

Para KNN y SVM es fundamental escalar las características.

In [ ]:
# Dividimos en train/test
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_digits, y_digits, test_size=0.3, random_state=42, stratify=y_digits
)

# Escalamos (importante para KNN y SVM)
scaler = StandardScaler()
X_train_d_scaled = scaler.fit_transform(X_train_d)
X_test_d_scaled = scaler.transform(X_test_d)

print(f"Tamaño de entrenamiento: {X_train_d_scaled.shape}")
print(f"Tamaño de prueba: {X_test_d_scaled.shape}")

## 3. KNN: Búsqueda del mejor k con Validación Cruzada

In [ ]:
# Definimos el rango de k a probar
k_range = list(range(1, 31, 2))  # impares de 1 a 30
k_scores = []

# Evaluamos cada k con validación cruzada
for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train_d_scaled, y_train_d, cv=5, scoring='accuracy')
    k_scores.append(scores.mean())

# Encontramos el mejor k
best_k = k_range[np.argmax(k_scores)]
best_score = max(k_scores)

print(f"Mejor k: {best_k} con accuracy promedio CV = {best_score:.4f}")

# Visualizamos
plt.figure(figsize=(10, 5))
plt.plot(k_range, k_scores, 'bo-')
plt.axvline(best_k, color='red', linestyle='--', label=f'Mejor k = {best_k}')
plt.xlabel('k')
plt.ylabel('Accuracy (CV)')
plt.title('KNN: Accuracy vs k')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Entrenamos el modelo con el mejor k
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train_d_scaled, y_train_d)
y_pred_knn = knn_best.predict(X_test_d_scaled)

print("=== KNN (mejor k) ===")
print(f"Accuracy en test: {accuracy_score(y_test_d, y_pred_knn):.4f}")
print("\nReporte de clasificación:")
print(classification_report(y_test_d, y_pred_knn))

# Matriz de confusión
plt.figure(figsize=(8, 6))
cm_knn = confusion_matrix(y_test_d, y_pred_knn)
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de Confusión - KNN')
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.show()

## 4. SVM con kernel RBF: Optimización de C y gamma

In [ ]:
# Definimos el grid de hiperparámetros
param_grid_svm = {
    'C': [0.1, 1, 10, 100],
    'gamma': [0.001, 0.01, 0.1, 1, 'scale', 'auto']
}

# Configuramos GridSearchCV
svm = SVC(kernel='rbf')
grid_svm = GridSearchCV(svm, param_grid_svm, cv=5, scoring='accuracy', verbose=1, n_jobs=-1)
grid_svm.fit(X_train_d_scaled, y_train_d)

print(f"Mejores parámetros SVM: {grid_svm.best_params_}")
print(f"Mejor accuracy CV: {grid_svm.best_score_:.4f}")

# Evaluamos en test
y_pred_svm = grid_svm.predict(X_test_d_scaled)
print(f"\nAccuracy en test: {accuracy_score(y_test_d, y_pred_svm):.4f}")

print("\nReporte de clasificación:")
print(classification_report(y_test_d, y_pred_svm))

# Matriz de confusión
plt.figure(figsize=(8, 6))
cm_svm = confusion_matrix(y_test_d, y_pred_svm)
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Greens')
plt.title('Matriz de Confusión - SVM')
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.show()

## 5. Comparación de tiempos de inferencia: KNN vs SVM

In [ ]:
# Medimos tiempo de predicción para KNN
start = time.time()
knn_best.predict(X_test_d_scaled)
time_knn = time.time() - start

# Medimos tiempo de predicción para SVM
start = time.time()
grid_svm.predict(X_test_d_scaled)
time_svm = time.time() - start

print(f"Tiempo de predicción KNN (mejor k): {time_knn:.6f} segundos")
print(f"Tiempo de predicción SVM (mejor modelo): {time_svm:.6f} segundos")

# Comparación visual
plt.figure(figsize=(8, 5))
plt.bar(['KNN', 'SVM'], [time_knn, time_svm], color=['blue', 'green'])
plt.ylabel('Tiempo (segundos)')
plt.title('Comparación de tiempos de inferencia')
plt.grid(axis='y')
plt.show()

---
# PARTE 2: CLASIFICACIÓN DE TEXTOS CON NAIVE BAYES

El dataset **20 Newsgroups** contiene aproximadamente 20,000 documentos agrupados en 20 categorías. Para simplificar, seleccionaremos 4 categorías y aplicaremos Multinomial Naive Bayes.

## 6. Carga y Preparación del Dataset 20 Newsgroups

In [ ]:
# Seleccionamos 4 categorías para un problema multiclase manejable
categories = ['alt.atheism', 'soc.religion.christian', 'comp.graphics', 'sci.space']

# Cargamos los datos (solo entrenamiento y prueba)
newsgroups_train = fetch_20newsgroups(subset='train', categories=categories, shuffle=True, random_state=42)
newsgroups_test = fetch_20newsgroups(subset='test', categories=categories, shuffle=True, random_state=42)

X_train_text = newsgroups_train.data
y_train_text = newsgroups_train.target
X_test_text = newsgroups_test.data
y_test_text = newsgroups_test.target

print(f"Categorías: {newsgroups_train.target_names}")
print(f"Tamaño entrenamiento: {len(X_train_text)}")
print(f"Tamaño prueba: {len(X_test_text)}")

# Mostramos un ejemplo de documento
print("\n=== Ejemplo de documento ===")
print(f"Categoría: {newsgroups_train.target_names[y_train_text[0]]}")
print(f"Texto:\n{X_train_text[0][:500]}...")

## 7. Vectorización con TF-IDF

Convertimos el texto a características numéricas usando TfidfVectorizer.

In [ ]:
# Vectorizador TF-IDF
vectorizer = TfidfVectorizer(stop_words='english', max_features=10000)

# Aprendemos el vocabulario y transformamos
X_train_tfidf = vectorizer.fit_transform(X_train_text)
X_test_tfidf = vectorizer.transform(X_test_text)

print(f"Dimensiones entrenamiento: {X_train_tfidf.shape}")
print(f"Dimensiones prueba: {X_test_tfidf.shape}")

## 8. Multinomial Naive Bayes: Entrenamiento y Evaluación

In [ ]:
# Entrenamos MultinomialNB
mnb = MultinomialNB()
mnb.fit(X_train_tfidf, y_train_text)

# Predicciones
y_pred_mnb = mnb.predict(X_test_tfidf)

print("=== Multinomial Naive Bayes ===")
print(f"Accuracy en test: {accuracy_score(y_test_text, y_pred_mnb):.4f}")
print("\nReporte de clasificación:")
print(classification_report(y_test_text, y_pred_mnb, target_names=newsgroups_train.target_names))

# Matriz de confusión
plt.figure(figsize=(8, 6))
cm_mnb = confusion_matrix(y_test_text, y_pred_mnb)
sns.heatmap(cm_mnb, annot=True, fmt='d', cmap='Purples', 
            xticklabels=newsgroups_train.target_names,
            yticklabels=newsgroups_train.target_names)
plt.title('Matriz de Confusión - MultinomialNB')
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 9. Análisis de las palabras más importantes por categoría

Una ventaja de Naive Bayes es que podemos inspeccionar las palabras con mayor probabilidad en cada clase.

In [ ]:
# Obtenemos el log-probabilidad de cada palabra por clase
feature_names = vectorizer.get_feature_names_out()
log_probs = mnb.feature_log_prob_

# Mostramos las 10 palabras más importantes para cada categoría
for i, category in enumerate(newsgroups_train.target_names):
    top_indices = np.argsort(log_probs[i])[::-1][:10]
    top_words = [feature_names[idx] for idx in top_indices]
    print(f"\nCategoría: {category}")
    print(f"Palabras más importantes: {', '.join(top_words)}")

---
## 10. Comparación Global de Resultados

Resumimos los resultados obtenidos en ambos problemas.

In [ ]:
# Resultados Digits
print("=== RESULTADOS DIGITS (clasificación de imágenes) ===")
print(f"KNN (k={best_k}): Accuracy = {accuracy_score(y_test_d, y_pred_knn):.4f}")
print(f"SVM RBF (C={grid_svm.best_params_['C']}, gamma={grid_svm.best_params_['gamma']}): Accuracy = {accuracy_score(y_test_d, y_pred_svm):.4f}")

print("\n=== RESULTADOS 20 NEWSGROUPS (clasificación de texto) ===")
print(f"Multinomial Naive Bayes: Accuracy = {accuracy_score(y_test_text, y_pred_mnb):.4f}")

# Discusión
print("\n=== DISCUSIÓN ===")
print("""
Observaciones:

1. **Digits**:
   - KNN y SVM obtienen accuracy muy alto (>98%).
   - SVM con kernel RBF suele igualar o superar ligeramente a KNN.
   - El tiempo de predicción de KNN es mayor que el de SVM una vez entrenado.

2. **20 Newsgroups**:
   - MultinomialNB es extremadamente rápido y efectivo en texto.
   - Accuracy alrededor de 90-95% en las categorías seleccionadas.
   - Podemos interpretar el modelo viendo las palabras más importantes.

3. **Elección del algoritmo**:
   - Para imágenes y datos numéricos, SVM suele ser superior.
   - Para texto, Naive Bayes es una opción natural y eficiente.
   - KNN es simple pero puede ser lento en predicción.
""")

---
## 11. Conclusiones

En este notebook hemos aplicado los algoritmos de clasificación clásicos a problemas reales:

✔️ **KNN en dígitos**: Encontramos el mejor k mediante validación cruzada y evaluamos su rendimiento.
✔️ **SVM en dígitos**: Optimizamos C y gamma con GridSearchCV, obteniendo un modelo muy preciso.
✔️ **Comparación de tiempos**: KNN es más lento en predicción que SVM.
✔️ **Naive Bayes en texto**: MultinomialNB logra alta precisión en clasificación de noticias.
✔️ **Interpretabilidad**: En Naive Bayes podemos inspeccionar las palabras más importantes por categoría.

**Lección clave**: No hay un algoritmo universalmente mejor; la elección depende del tipo de datos, la necesidad de interpretabilidad y los requisitos de velocidad.

---
**Fin del Notebook de Ejercicios - Semana 5**